In [ ]:
!pip install --upgrade pip
!pip install unsloth
!pip install transformers==4.55.4 trl==0.22.2
from google.colab import files
import pandas as pd
import json
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt
from unsloth import FastLanguageModel
import torch
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Upload dataset
uploaded = files.upload()

# Load JSONL file
df = pd.read_json("diabetes_dataset_for_llm_fine_tuning.jsonl", lines=True)

print("✅ File loaded successfully!")
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
df.head()

Saving diabetes_dataset_for_llm_fine_tuning.jsonl to diabetes_dataset_for_llm_fine_tuning (1).jsonl
✅ File loaded successfully!
Columns: ['question', 'answer']
Shape: (5000, 2)


,question,answer
0,What is diabetes?,Diabetes is a chronic disease where blood suga...
1,What is diabetes?,Diabetes is a chronic disease where blood suga...
2,What are the main types of diabetes?,"The main types are type 1 diabetes, type 2 dia..."
3,What are GLP-1 receptor agonists?,GLP-1 receptor agonists are medicines that imp...
4,What symptoms appear in children with diabetes?,"Children may show excessive thirst, bedwetting..."


In [ ]:
output_path = "llama3_conversations.jsonl"

# Convert to ShareGPT JSONL
with open(output_path, "w", encoding="utf-8") as f:
    for idx, row in df.iterrows():
        convo = {
            "conversations": [
                {"from": "human", "value": str(row["question"]).strip()},
                {"from": "gpt", "value": str(row["answer"]).strip()}
            ]
        }
        f.write(json.dumps(convo, ensure_ascii=False) + "\n")

print(f"✅ Saved {len(df)} conversation pairs to {output_path}")

✅ Saved 5000 conversation pairs to llama3_conversations.jsonl


In [ ]:
dataset = load_dataset("json", data_files=output_path, split="train")

def to_sharegpt_format(example):
    return {
        "conversations": [
            {"from": "human", "value": example["conversations"][0]["value"]},
            {"from": "gpt", "value": example["conversations"][1]["value"]}
        ]
    }

dataset = dataset.map(to_sharegpt_format)
dataset = standardize_sharegpt(dataset)

print("✅ Dataset loaded and standardized. Samples:", len(dataset))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

✅ Dataset loaded and standardized. Samples: 5000


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=1024,
    dtype="float16",
    load_in_4bit=True
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✅ Base model ready for LoRA fine-tuning.")

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Base model ready for LoRA fine-tuning.


In [ ]:
prompt = "What is metaformin"

# Tokenize the input
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate a reply
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,   # how long the reply can be
        do_sample=True,       # enables sampling for more natural replies
        temperature=0.7,      # controls randomness
        top_p=0.9             # nucleus sampling
    )

# Decode the reply
reply = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Model reply:", reply)

Model reply: What is metaformin?
Metaformin is a medication used to treat type 2 diabetes. It works by increasing the sensitivity of the body's cells to insulin, which helps to lower blood sugar levels.

What are the benefits of metaformin?
The benefits of metaformin include:

* Reduced risk of cardiovascular disease and stroke
* Improved blood sugar control
* Weight loss with a healthy diet and exercise
* Reduced risk of kidney damage and other complications of diabetes
* Improved quality of life

How does metaformin work?
Metaformin works by binding to the α2C subunit of the ATP-sensitive potassium channel in the liver and pancreas. This binding causes a change in the electrical properties of the cell membrane, which increases insulin sensitivity and improves glucose metabolism.

Who is a good candidate for metaformin?
Metaformin may be suitable for patients with type 2 diabetes who have:

* Normal kidney function
* Normal liver function
* No severe kidney disease
* No severe cardiov

In [ ]:
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.2")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

print("✅ Dataset formatted with chat template.")


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

✅ Dataset formatted with chat template.


In [ ]:

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    data_collator=DataCollatorForSeq2Seq(tokenizer),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,  # Increase for full training
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        save_strategy="steps",
        save_steps=20,
        save_total_limit=2
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

trainer.train()

print("✅ Training complete.")

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/5000 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/5000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 5,636,096 of 1,241,450,496 (0.45% trained)


Step,Training Loss
1,2.792200
2,2.611800
3,2.344900
4,2.121700
5,1.957500
6,1.479800
7,1.262800
8,1.098500
9,1.060700
10,0.612900


✅ Training complete.


In [ ]:
# Save LoRA adapter and tokenizer
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

print("✅ LoRA adapter and tokenizer saved in 'lora_model/'")

# Load base model with offloading to disk (prevents RAM overflow)
base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Llama-3.2-1B-Instruct",
    device_map="auto",              # spreads across CPU/GPU
    offload_folder="offload"        # spills tensors to disk
)

# Load LoRA adapter into base model
lora_model = PeftModel.from_pretrained(base_model, "lora_model")

# Merge LoRA weights into base model
merged_model = lora_model.merge_and_unload()

# Save merged model and tokenizer
merged_model.save_pretrained("merged_model")
tokenizer = AutoTokenizer.from_pretrained("lora_model")
tokenizer.save_pretrained("merged_model")

print("✅ Merged model saved in 'merged_model/'. Ready for GGUF conversion.")

✅ LoRA adapter and tokenizer saved in 'lora_model/'
✅ Merged model saved in 'merged_model/'. Ready for GGUF conversion.


In [ ]:
!zip -r lora_model.zip lora_model
from google.colab import files
files.download("lora_model.zip")

updating: lora_model/ (stored 0%)
updating: lora_model/adapter_config.json (deflated 58%)
updating: lora_model/README.md (deflated 65%)
updating: lora_model/chat_template.jinja (deflated 72%)
updating: lora_model/tokenizer_config.json (deflated 96%)
updating: lora_model/adapter_model.safetensors (deflated 7%)
updating: lora_model/special_tokens_map.json (deflated 71%)
updating: lora_model/tokenizer.json (deflated 85%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 78467, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (176/176), done.
remote: Total 78467 (delta 157), reused 63 (delta 63), pack-reused 78228 (from 4)
Receiving objects: 100% (78467/78467), 287.52 MiB | 34.90 MiB/s, done.
Resolving deltas: 100% (56752/56752), done.


In [ ]:
!python3 llama.cpp/convert_hf_to_gguf.py merged_model --outfile diabetes_gguf --outtype q8_0

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {32}
INFO:hf-to-gguf:token_embd.weight,           torch.float32 --> Q8_0, shape = {2048, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float32 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float32 --> Q8_0, shape = {8192, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float32 --> Q8_0, shape = {2048, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float32 --> Q8_0, shape = {2048, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float32 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float32 --> Q8_0, shape = {2048, 512}
INFO:hf-to-gguf:blk.0.attn_output.weigh

In [ ]:
files.download("diabetes_gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>